# Introduction

This notebook documents the current signal processing pipeline.


# Load Python modules


In [ ]:
import time

import numpy as np

# Plotting functions
import plotly.graph_objects as go

# Force Plotly figure to be shown in jupyterlab
import plotly.io as pio
from IPython.utils.capture import capture_output
from matplotlib import pyplot as plt

from mascope_file.io import load_coord

# Data loading
from mascope_file.name import get_instrument_type
from mascope_signal.compute import get_sum_signal


pio.renderers.default = "jupyterlab"

# Resolution function fitting
from lmfit.models import (
    SkewedGaussianModel,
    SplineModel,
)
from scipy.optimize import curve_fit

# Peak fitting
from scipy.signal import find_peaks, peak_widths
from scipy.stats import linregress

from mascope_signal.peak import detect_peaks, fwhm_to_sigma, gen_peak

# Load file


In [ ]:
file_name = "KORBI2_20240712_Pesticide_apple_161310_-"
# Define the MS
instrument_type = get_instrument_type(file_name)
# Extract averaged spectrum and mz values
spec = get_sum_signal(file_name).values
mz = load_coord(file_name, "sum_signal", "mz")
# Smooth spectrum for better fitting if needed
# scales = np.arange(0.01, 1, 0.01)
# spec = mgspe(spec, scales)
# Find peaks. Tune the height if there are no peaks/too many peaks detected.
# The higher np.percentile 2nd parameter values, the higher peak height threshold
peak, _ = find_peaks(spec, height=np.percentile(spec, 95.))
fwhm = peak_widths(spec, peak, rel_height=0.5)[0]
sig = np.array([fwhm_to_sigma(fw) for fw in fwhm])

# Plot result
fig = go.Figure(
    [
        go.Scatter(x=mz, y=spec, name="Raw counts"),
        go.Scatter(x=mz[peak], y=spec[peak], mode="markers", name="Detected peaks"),
    ]
)
fig.update_layout(
    height=300, width=800, margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4)
)
print(f"{len(peak)} peaks were detected.")
fig.show()

# Peak shape estimation


In [ ]:
def fit_gaussian(x, y):
    """Fit the spectrum range with a skewed Gaussian peak-shape using lmfit

    :param x: mz scale
    :type x: array-like
    :param y: counts
    :type y: array-like
    :return: ModelResult (see lmfit doc)
    :rtype: ModelResult
    """
    # Initialize fitting parameters for the main peak
    model = SkewedGaussianModel(prefix="p_")
    params = model.make_params()
    if instrument_type == "tof":
        # Fitting parameters for the background
        knot_xvals = np.array([-dmz, -dmz / 2, -dmz / 3, dmz / 3, dmz / 2, dmz])
        bkg = SplineModel(prefix="bkg_", xknots=knot_xvals)
        params.update(bkg.guess(y, x))
        # Total model
        model = model + bkg

    params.add("p_amplitude", value=1, max=1.1)
    params.add("p_center", value=0)
    params.add("p_sigma", value=0.01)
    params.add("p_gamma", value=0.1)

    # Perform fitting
    fit = model.fit(y, params, x=x)
    return fit


def calculate_fwhm(x, y):
    # Find the maximum count and its index
    max_index = np.argmax(y)
    max_count = y[max_index]

    # Find the half maximum count
    half_max_count = max_count / 2

    # Find the indices where the counts are closest to the half maximum
    left_index = np.argmin(np.abs(y[:max_index] - half_max_count))
    right_index = np.argmin(np.abs(y[max_index:] - half_max_count)) + max_index

    # Calculate the FWHM
    fwhm = x[right_index] - x[left_index]

    return fwhm


def calculate_center_of_mass(x, y):
    # Calculate the center of mass
    center_of_mass = np.sum(x * y) / np.sum(y)
    return center_of_mass

In [ ]:
# Expected half-width band presumably containing one peak
# Decrease dmz if you get too many peaks in one band
# Increase dmz if the peak shape is distorted/cut
dmz = 0.1  # TODO find way to automatically estimate optimal dmz value
r_squared_threshold = 0.9

peak_traces = []
peak_norm_traces = []

p_x = np.linspace(-10, 10, 101)
p_ys = []
p_mzs = []
p_fwhms = []
p_centers = []

for i, p in enumerate(peak):
    p_height = spec[p]
    # Select a narrow region (peak center +/- dmz) of the spectrum around the peak
    p_mz_center = mz[p]
    p_mz0 = p_mz_center - dmz
    p_mz1 = p_mz_center + dmz
    # Find indices
    p_i0 = np.argmin(np.abs(mz - p_mz0))  # left border
    p_i1 = np.argmin(np.abs(mz - p_mz1))  # right border
    p_i = p - p_i0  # center
    # Peak region
    p_mz = mz[p_i0:p_i1]
    p_spec = spec[p_i0:p_i1]

    if np.max(p_spec) > p_height:
        # 'p' is not the biggest peak in range, dismiss
        continue

    # Normalize peak region: mz around 0 and spec to range [0, 1]
    p_mz_norm = p_mz - p_mz_center
    p_spec_norm = p_spec / p_height
    p_spec_norm -= p_spec_norm.min()

    peak_traces.append(go.Scatter(x=p_mz_norm, y=p_spec_norm))
    # Fit peak in the region
    fit = fit_gaussian(p_mz_norm, p_spec_norm)
    p_spec_norm_fit = fit.eval_components()["p_"]

    if fit.rsquared < r_squared_threshold:
        # fitting error to large, dismiss
        continue

    # Remove junk peaks arond main one
    mask = np.where(fit.best_fit < 1e-9)
    p_spec_norm[mask] = 0

    # Get peak top location

    if instrument_type == "tof":
        # Remove baseline if TOF
        p_spec_norm -= fit.eval_components()["bkg_"]
        p_spec_norm[np.where(p_spec_norm < 0)] = 0

    top_y = np.max(p_spec_norm_fit)
    top_x = calculate_center_of_mass(p_mz_norm, p_spec_norm_fit)

    # Refine peak position and height
    p_mz_norm -= top_x
    p_spec_norm /= top_y

    # if np.abs(p_mz_norm[np.argmin(np.abs(p_spec_norm - np.max(p_spec_norm)))]) > 5e-3:
    #     continue

    # Get and store Gaussian peak sigma and width
    try:
        p_fwhm = calculate_fwhm(p_mz_norm, p_spec_norm_fit)
    except Exception:
        continue

    # p_sigma = fit.params["p_sigma"].value
    p_sigma = p_fwhm / (2 * np.sqrt(2 * np.log(2)))
    # Scale peak to width sigma=1
    p_mz_norm /= p_sigma
    # Interpolate the normalized (both width and height) peak into predefined domain "p_x"
    p_y = np.interp(p_x, p_mz_norm, p_spec_norm, left=0, right=0)
    if np.all(np.isnan(p_y)):
        continue
    # Store peak positions, widths, and refined peak shape
    p_mzs.append(p_mz_center)
    p_centers.append(top_x)
    p_fwhms.append(p_fwhm)
    p_ys.append(p_y)
    peak_norm_traces.append(go.Scatter(x=p_x, y=p_y, name=f"peak {i}"))

# Clean shifted outliers
p_centers = np.array(p_centers)
non_outlier_indx = np.where(p_centers - np.median(p_centers) < 1e-3)[0]
peak_norm_traces = [peak_norm_traces[i] for i in non_outlier_indx]
p_fwhms = [p_fwhms[i] for i in non_outlier_indx]
p_ys = [p_ys[i] for i in non_outlier_indx]
p_mzs = [p_mzs[i] for i in non_outlier_indx]

# Calculate median peak shape
p_median = np.median(np.array([p_y for p_y in p_ys]), axis=0)
# Check if p_median is empty
if p_median.all() == np.nan:
    raise Exception(
        """p_median is empty
        Probably fitting error threshold is too strict"""
    )
# Add to the figure
peak_norm_traces.append(
    go.Scatter(x=p_x, y=p_median, line={"color": "red", "width": 3}, name="median")
)
# Plot the output
fig = go.FigureWidget(
    peak_norm_traces,
    {"title": f"Median peak shape of {len(peak_norm_traces)-1} peaks"},
)
fig.update_layout(
    height=300, width=800, margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4)
)


def toggle_trace_visibility(trace, points, selector):
    """Updates the median peak shape based on shapes present in the figure"""
    global p_median
    if not points.point_inds:
        return
    trace.visible = "legendonly"

    selected_traces = [
        trace.y
        for trace in fig.data
        if ("peak" in trace.name and trace.visible != "legendonly")
    ]

    p_median = np.median(np.array(selected_traces), axis=0)

    fig.update_layout(
        title=f"Median peak shape of {len(selected_traces)} peaks",
        height=300,
        width=800,
        margin=go.layout.Margin(l=30, r=30, b=30, t=30, pad=4),
    )
    fig.update_traces(
        {"y": p_median},
        selector=lambda trace: trace.name == "median",
    )


print("Click on distorted peaks to remove them from peak shape estimation.")
[trace.on_click(toggle_trace_visibility) for trace in fig.data]
fig

Save peak shape and check how good the mean shape align with peaks along mz scale, R-squared used as error estimation.


In [ ]:
# Save peak
peak_shape = {"x": p_x, "y": p_median}
# To JSON file if needed
# with open(f"C:/Users/alvai/Desktop/peak_shapes/{file_name}_peak_shape.json", "w") as f:
#     json.dump({"x": list(p_x), "y": list(p_median)}, f)
p_ys = np.array(p_ys)
# Calculate residuals for normalized peaks vs median peak shape
norm_residuals = p_ys - peak_shape["y"]
## calculate R-squared vs mz
r_sq_norm = 1 - (norm_residuals**2).sum(axis=1) / ((p_ys - p_ys.mean()) ** 2).sum(
    axis=1
)
# Plot error vs mz
err_fig = plt.figure(figsize=(7, 1.5))
plt.scatter(p_mzs, r_sq_norm)
plt.xlabel("mz")
plt.ylabel(r"R$^2$")

## Resolution function

In [ ]:
# Resolution functions
R_tof = lambda mz: mz / (tof_a * mz + tof_b)
R_orb = lambda mz: orb_a / np.sqrt(mz)
# Fittin support functions
line = lambda x, a, b: a * x + b
polynome = lambda x, a, b, c: a * x**2 + b * x + c
rational_polynome = lambda x, a, b: x / (a * x + b)
inverse_sqrt = lambda x, a: a / np.sqrt(x)

Fitting of the resolution function


In [ ]:
p_mzs = np.array(p_mzs)
# Get the fitted line depending on the instrument
if instrument_type == "tof":
    regres = linregress(p_mzs, p_fwhms)
    p_fwhms_line = line(p_mzs, regres.slope, regres.intercept)
if instrument_type == "orbi":
    coefs = np.polyfit(p_mzs, p_fwhms, 2)
    p_fwhms_line = polynome(p_mzs, *coefs)

# Points higher than ndev * std_dev will be later filtered out
ndev = 1

# Get residuals and standard deviation
residuals = p_fwhms - p_fwhms_line
std_dev = np.std(residuals)
is_outlier = (residuals > ndev * std_dev) | (residuals < -ndev * std_dev)

# Remove outliers
p_fwhms_filt = np.array(p_fwhms, dtype=np.double)[~is_outlier]
mass = np.array(p_mzs, dtype=np.double)[~is_outlier]

# mz filter (optional)
mass_filter = mass < 3000
mass = mass[mass_filter]
p_fwhms_filt = np.array(p_fwhms, dtype=np.double)[~is_outlier][mass_filter]
resolution = mass / p_fwhms_filt

if instrument_type == "tof":
    # Fit resolution with the inverse square root
    try:
        # TOF initial guesses
        a_init = 1 / np.mean(resolution)
        b_init = 0
        bounds = ([-np.inf, 0], [np.inf, np.inf])
        # Resolution uncertainties
        resolution_uncertainties = resolution * std_dev * p_fwhms_filt
        popt, pcov = curve_fit(
            rational_polynome,
            mass,
            resolution,
            sigma=resolution_uncertainties,
            p0=(a_init, b_init),
            bounds=bounds,
        )
        tof_a, tof_b = popt
    except ValueError as e:
        print(e)
        tof_a, tof_b = a_init, b_init
    resolution_function = R_tof

    print(f"TOF, a={tof_a:.2e}, b={tof_b:.2e}")

if instrument_type == "orbi":
    # Fit resolution with the inverse square root
    try:
        popt, pcov = curve_fit(inverse_sqrt, mass, resolution)
        orb_a = popt[0]
    except ValueError as e:
        print(e)
        orb_a = 1.725e6
    resolution_function = R_orb

    print(f"Orbi, a={orb_a:.2e}")

# Plot figure to see outliyers
fig, (ax_res, ax_fwhm) = plt.subplots(nrows=1, ncols=2, figsize=(12, 3))
ax_fwhm.plot(p_mzs, p_fwhms, label="True FWHM")
ax_fwhm.plot(p_mzs, p_fwhms_line, linestyle="--", color="red", label="Approximation")
ax_fwhm.fill_between(
    p_mzs,
    p_fwhms_line + ndev * std_dev,
    p_fwhms_line - ndev * std_dev,
    alpha=0.2,
    label="Non-outliers",
)
ax_fwhm.set(ylabel="FWHM", xlabel="mz")
ax_fwhm.legend()

# Plot fit
ax_res.scatter(
    p_mzs, np.array(p_mzs) / np.array(p_fwhms), color="grey", label="Ommited pairs"
)
ax_res.scatter(mass, resolution, color="black", label="Used mass/resolution pairs")

mass_range = np.linspace(min(p_mzs), max(p_mzs), 100)

ax_res.plot(
    mass_range, resolution_function(mass_range), "r", label="Fitted resolution function"
)
ax_res.set(
    ylabel="Resolution",
    xlabel="mz",
    title="Resolution function",
)
ax_res.legend()

# Peak fitting


In [ ]:
# Assign peak fitting threshold depending on the instrument type
# Correct intrument type unsured by get_instrument_type
if instrument_type == "orbi":
    threshold = 0.8
if instrument_type == "tof":
    threshold = 0.9
# Rich output may hang the system, that's why we catch it and save in cap
execution_times = []
for _ in range(3):
    with capture_output() as cap:
        start_time = time.time()
        sample_file_data, all_peak_mzs = await detect_peaks(
            file_name,
            (peak_shape, resolution_function),
            threshold,
            # u_list=np.arange(200, 300, 1),  # for testing
            max_n_peaks=5,
            if_exists="replace",
            dmz=0.5,
            return_peak_mzs=True,
            instrument_type=instrument_type,
        )
        end_time = time.time()
        execution_times.append(end_time - start_time)
# NOTE all the outputs can be seen with cap.show()
# the list of outputs accessed with cap.outputs
# Calculate mean and standard deviation
mean_execution_time = np.mean(execution_times)
std_execution_time = np.std(execution_times)

print(f"Execution time: {mean_execution_time:.3f} ± {std_execution_time:.3f} seconds")


In [ ]:
from pythonnet import load


load("coreclr")
import sys

import clr


sys.path.append(r"C:\Users\alvai\Desktop\PeakLibrary\bin\Debug\net8.0")
clr.AddReference("PeakLibrary")
from PeakLibrary import PeakGenerator


# Create an instance of the C# class
peak_generator = PeakGenerator()

# Define your parameters
x = np.linspace(0, 10, 100)
ppos = 5.0
phei = 1.0
pres = 1.0
ps = {'x': np.linspace(0, 1, 100), 'y': np.random.normal(size=100)}
sigma_multiplier = 1.0  # Replace with actual value

# Convert numpy arrays to lists for C# compatibility
x_list = x.tolist()
ps_x_list = ps['x'].tolist()
ps_y_list = ps['y'].tolist()

# Call the C# method
start_time = time.time()
peak = peak_generator.GeneratePeak(x_list, ppos, phei, pres, peak_shape["x"], peak_shape["y"], sigma_multiplier)
print(f"{time.time() - start_time}")

# Convert the result back to a numpy array
peak = np.array(peak)

# Print the result
plt.plot(x_list, peak)

In [ ]:
start_time = time.time()
a = gen_peak(x_list, ppos, phei, pres, peak_shape)
print(f"{time.time() - start_time}")

In [ ]:
start_time = time.time()
a = peak_generator.GeneratePeak(x_list, ppos, phei, pres, peak_shape["x"], peak_shape["y"], sigma_multiplier)
print(f"{time.time() - start_time}")